# Verify WidowX robot control

Verify the current machine's native WidowX binding and Interbotix connection, make small arm movements, and return to the Interbotix **sleep pose** (rest position).

Run cells in order with the Interbotix ROS environment available and the robot driver running. Start from a clear, reachable working pose; a folded sleep pose may not permit these Cartesian tests. Clear the test paths and the path to sleep: bounds and inverse-kinematics checks do not detect collisions.

**Note** that the robot sometimes fails in the calculations in returning to the initial position where it was, as it appears to be unsafe for it. 


In [1]:
import sys
sys.path.append("..")

from copy import copy, deepcopy
from time import sleep
from exp_run_config import Config, Experiment
from robot.widowx import PositionController, WidowXRuntime

Config.PROJECTNAME = "BerryPicker"
machine_exp = Config().get_experiment("machine", "current")
binding = machine_exp["bindings"]["widowx_robot"]
if not binding["available"]:
    raise RuntimeError("The widowx_robot machine binding is unavailable")
if binding["factory"] != "widowx_hardware":
    raise ValueError("Verification requires the native Interbotix WidowX binding")
exp_robot_controller = Config().get_experiment(binding["exp"], binding["run"])
print(f"Machine: {machine_exp['machine_name']}")
print(f"Robot configuration: {binding['exp']}/{binding['run']}")
for field in ("robot_model", "interbotix_robot_name", "group_name", "gripper_name"):
    print(f"{field}: {exp_robot_controller[field]}")
print("Configured startup/shutdown:",
      exp_robot_controller["startup_pose"], exp_robot_controller["shutdown_pose"])


***ExpRun**: Loading pointer config file:
	/home/lboloni/.config/BerryPicker/mainsettings.yaml
***ExpRun**: Loading machine-specific config file:
	/home/lboloni/Insync/lotzi.boloni@gmail.com/Google Drive/LotziStudy/Code/PackageTracking/BerryPicker/settings/settings-tredy2.yaml
***ExpRun**: Using torch device: cuda
***ExpRun**: Experiment default config /home/lboloni/Documents/Hackingwork/_Checkouts/BerryPicker/BerryPicker/src/experiment_configs/machine/_defaults_machine.yaml was empty, ok.
***ExpRun**: Configuration for exp/run: machine/current successfully loaded
***ExpRun**: Experiment default config /home/lboloni/Documents/Hackingwork/_Checkouts/BerryPicker/BerryPicker/src/experiment_configs/robot_widowx/_defaults_robot_widowx.yaml was empty, ok.
***ExpRun**: Configuration for exp/run: robot_widowx/position_controller_wx250s_00 successfully loaded
Machine: tredy2
Robot configuration: robot_widowx/position_controller_wx250s_00
robot_model: wx250s
interbotix_robot_name: wx250s
group_n

## Enable motion deliberately

Set ALLOW_ROBOT_MOTION to True only after checking the workspace. Each slow, blocking test returns to the measured starting pose. The final parking movement can be larger than the test offsets. The gripper is left unchanged.

A private copy of the exp sets startup to hold and shutdown to sleep, without changing saved configuration.


In [2]:
ALLOW_ROBOT_MOTION = True
MOVING_TIME = 3.0
test_offsets = {
    "z": 0.02,       # 1 cm upward
}
verification_exp = Experiment(deepcopy(exp_robot_controller.values))
# Update the private dictionary without triggering Experiment.__setitem__ saving it.
verification_exp.values.update({
    "startup_pose": "hold",
    "shutdown_pose": "sleep",
    "moving_time": MOVING_TIME,
    "accel_time": 0.5,
})


In [3]:
if not ALLOW_ROBOT_MOTION:
    raise RuntimeError("Set ALLOW_ROBOT_MOTION = True after checking the workspace")

# A fresh runtime allows repeating this cell after a completed shutdown.
rob = PositionController(verification_exp, runtime=WidowXRuntime())
try:
    rob.start_robot()
    start_position = rob.get_position()
    print("Connected. Measured starting pose:")
    print(start_position)
    print("Initial telemetry:", rob.get_state())

    # Validate every target before issuing the first test movement.
    targets = []
    for field, offset in test_offsets.items():
        target = copy(start_position)
        target[field] += offset
        target.validate(verification_exp)
        if not rob.can_reach(target):
            raise ValueError(f"Unreachable verification target: {field} offset {offset}")
        targets.append((field, offset, target))
    if not rob.can_reach(start_position):
        raise ValueError("Cannot return to the measured starting pose")

    for field, offset, target in targets:
        print(f"Testing {field}: offset {offset} (meters or radians)")
        rob.move(target, moving_time=MOVING_TIME, blocking=True)
        print("Measured pose after movement:")
        print(rob.get_position())
        sleep(1.0)
        rob.move(start_position, moving_time=MOVING_TIME, blocking=True)
        sleep(1.0)
    print("Verification movements completed. Confirm the observed motion was correct.")
finally:
    # Attempt to park even after a failed test or keyboard interrupt.
    # Parking failures propagate; do not report success if parking fails.
    if rob.started:
        rob.stop_robot()
        print("Returned to the sleep pose and shut down the ROS runtime.")


[INFO] [1789142797.012901904] [interbotix_robot_manipulation]: Initialized InterbotixRobotNode!
[INFO] [1789142797.055833515] [interbotix_robot_manipulation]: 
	Robot Name: wx250s
	Robot Model: wx250s
[INFO] [1789142797.056315566] [interbotix_robot_manipulation]: Initialized InterbotixRobotXSCore!
[INFO] [1789142797.063890837] [interbotix_robot_manipulation]: 
	Arm Group Name: arm
	Moving Time: 3.00 seconds
	Acceleration Time: 0.50 seconds
	Drive Mode: Time-Based-Profile
[INFO] [1789142797.064371245] [interbotix_robot_manipulation]: Initialized InterbotixArmXSInterface!
[INFO] [1789142797.570816588] [interbotix_robot_manipulation]: 
	Gripper Name: gripper
	Gripper Pressure: 50.0%
[INFO] [1789142797.571829583] [interbotix_robot_manipulation]: Initialized InterbotixGripperXSInterface!


Connected. Measured starting pose:
WidowX pose:
 x: 0.1229
 y: -0.0012
 z: 0.0718
 roll: -0.0074
 pitch: 0.5522
 yaw: -0.0093

Initial telemetry: {'timestamp': 4473.517667072, 'pose': {'x': 0.12296218579671361, 'y': -0.0012379012753693458, 'z': 0.07234667496585587, 'roll': -0.007392544077819659, 'pitch': 0.5506645061596331, 'yaw': -0.009298784988227362}, 'joint_positions': [-0.0015339808305725455, -1.8361750841140747, 1.5846022367477417, -0.00920388475060463, 0.8022719621658325, 0.003067961661145091], 'gripper_action': 'hold', 'gripper_position': 0.014176830649375916}
Testing z: offset 0.02 (meters or radians)
Measured pose after movement:
WidowX pose:
 x: 0.1223
 y: -0.0016
 z: 0.0798
 roll: -0.0110
 pitch: 0.5598
 yaw: -0.0119

Verification movements completed. Confirm the observed motion was correct.
Returned to the sleep pose and shut down the ROS runtime.


## After verification

A completed run ends in sleep. Shutdown stops the ROS runtime; it does not claim to disable motor torque. If motion or parking raises an exception, inspect the error and robot state before retrying. A killed kernel or lost connection can prevent the finally block from parking the arm.
